# Lesson 7.2 — Targeted Responses: Matching Action to Owner

The recovery policy maps each fault to a targeted response routed by owner, tagged retryable (transient) or deterministic.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, recover, RECOVERY_POLICY)
W = GreenhouseWorld.demo_row(n=6, seed=1)
ALL_IDS = [f.fid for f in W.fruit]
TGT = understand(model_perception(W, rng=np.random.default_rng(7)), W)[1]
def kick(t, j): return 60.0 if (j == 0 and t > 0.3) else 0.0
checks = []
for code in ('NO_TARGET','TRACKING_FAILURE','UNREACHABLE','PLAN_INVALID'):
    p = RECOVERY_POLICY[code]
    print('%-17s -> owner=%-22s response=%-16s retryable=%s' % (code, p['owner'], p['response'], p['retryable']))
    checks.append(all(k in p for k in ('owner','response','retryable')))


NO_TARGET         -> owner=Perceive / Understand  response=re-perceive      retryable=True
TRACKING_FAILURE  -> owner=Execute                response=retry-execute    retryable=True
UNREACHABLE       -> owner=Understand             response=re-select-target retryable=False
PLAN_INVALID      -> owner=Plan                   response=skip-target      retryable=False


### Retryable (transient) vs deterministic

In [2]:
checks.append(RECOVERY_POLICY['NO_TARGET']['retryable'] is True)
checks.append(RECOVERY_POLICY['TRACKING_FAILURE']['retryable'] is True)
checks.append(RECOVERY_POLICY['UNREACHABLE']['retryable'] is False)
checks.append(RECOVERY_POLICY['PLAN_INVALID']['retryable'] is False)


### The policy routes each fault: transient occlusion re-perceives; blocked goal skips

In [3]:
r_occ = recover(W, W.q.copy(), perception=lambda a: dict(occlude=ALL_IDS) if a == 0 else {})
print('transient occlusion -> success=%s recovered=%s (re-perceived)' % (r_occ['success'], r_occ['recovered']))
checks.append(r_occ['success'] and r_occ['recovered'])
r_blk = recover(W, W.q.copy(), obstacle=(TGT['xy'], 0.25))
print('blocked goal        -> escalated=%s n_attempts=%d (deterministic skip)' % (r_blk['escalated'], r_blk['n_attempts']))
checks.append(r_blk['escalated'] and r_blk['n_attempts'] == 1)
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


transient occlusion -> success=True recovered=True (re-perceived)


blocked goal        -> escalated=True n_attempts=1 (deterministic skip)
10/10 checks passed.
All checks passed.
